<a href="https://colab.research.google.com/github/RIDDHI1624/Drug-Discovery/blob/main/Insulin_Receptor_Project/Phase3_3.3_step3_equilibration_cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3.3 Step 3: Equilibration (CPU Version)
**Steps covered:**
- Step 16: NVT equilibration (100 ps) — heat to 310 K, V-rescale thermostat
- Step 17: NPT equilibration (100 ps) — Parrinello-Rahman barostat at 1 bar
- Step 18: Restraint release (optional)

**Expected time on CPU:**
- NVT: ~45-90 min
- NPT: ~45-90 min

**All known issues pre-fixed:**
- posre.itp auto-generated
- Standard tc-grps (Protein / Non-Protein)
- -r flag in all grompp calls
- Checkpoint resume if session interrupted

In [1]:
# Cell 0 — Install GROMACS + imports
import subprocess, zipfile, glob, re, os, shutil
from pathlib import Path

print('Installing GROMACS...')
subprocess.run(['apt-get', 'install', '-y', 'gromacs'], capture_output=True)

ver = subprocess.run(['gmx', '--version'], capture_output=True, text=True)
if ver.returncode == 0:
    lines = [l for l in ver.stdout.splitlines() if 'GROMACS version' in l]
    print(f'✓ {lines[0].strip()}' if lines else '✓ GROMACS installed')
else:
    raise RuntimeError('GROMACS install failed')

GMX = 'gmx'
print(f'✓ GMX = "{GMX}"')

Installing GROMACS...
✓ GROMACS version:    2021.4-Ubuntu-2021.4-2
✓ GMX = "gmx"


In [2]:
# Cell 1 — Upload files
from google.colab import files as colab_files

zips = glob.glob('/content/*.zip')
em_exists = Path('/content/em.gro').exists()

if not zips:
    print('Upload gromacs_system_candidate_0004.zip:')
    colab_files.upload()
    zips = glob.glob('/content/*.zip')
else:
    print(f'✓ Zip already present: {Path(zips[0]).name}')

if not em_exists:
    print('Upload em.gro (from Step 2 output):')
    colab_files.upload()
else:
    print('✓ em.gro already present')

Upload gromacs_system_candidate_0004.zip:


Saving gromacs_system_candidate_0004.zip to gromacs_system_candidate_0004.zip
Upload em.gro (from Step 2 output):


Saving system.gro to system.gro


In [4]:
# Cell 2 — Extract files from zip
zips = glob.glob('/content/*.zip')
if not zips:
    raise FileNotFoundError('No zip found — run Cell 1 first')
zip_path = zips[0]
print(f'✓ Using: {Path(zip_path).name}')

with zipfile.ZipFile(zip_path, 'r') as zf:
    for fname in ['system.gro', 'topol.top', 'complex.top']:
        if fname in zf.namelist():
            zf.extract(fname, '/content/')
            print(f'✓ Extracted {fname}')
        else:
            print(f'✗ {fname} not in zip')

if not Path('/content/em.gro').exists():
    raise FileNotFoundError('em.gro not found — upload from Step 2')

with open('/content/em.gro', 'r') as f:
    f.readline()
    n_atoms = int(f.readline().strip())
print(f'\n✓ em.gro found — n_atoms = {n_atoms}')

✓ Using: gromacs_system_candidate_0004.zip
✓ Extracted system.gro
✓ Extracted topol.top
✓ Extracted complex.top

✓ em.gro found — n_atoms = 40988


In [5]:
# Cell 3 — Generate posre.itp files
os.makedirs('/content/gromacs_md', exist_ok=True)

# Full restraints — 1000 kJ/mol/nm² all heavy atoms (Steps 16+17)
r1 = subprocess.run(
    f'echo "Protein" | {GMX} genrestr -f /content/em.gro -o /content/posre.itp -fc 1000 1000 1000',
    shell=True, capture_output=True, text=True
)
if Path('/content/posre.itp').exists():
    shutil.copy('/content/posre.itp', '/content/gromacs_md/posre.itp')
    print('✓ posre.itp generated (1000 kJ/mol/nm² all heavy atoms)')
else:
    raise RuntimeError(f'posre.itp failed: {r1.stderr[-200:]}')

# Backbone-only restraints — 100 kJ/mol/nm² (Step 18)
r2 = subprocess.run(
    f'echo "Backbone" | {GMX} genrestr -f /content/em.gro -o /content/posre_backbone.itp -fc 100 100 100',
    shell=True, capture_output=True, text=True
)
if Path('/content/posre_backbone.itp').exists():
    shutil.copy('/content/posre_backbone.itp', '/content/gromacs_md/posre_backbone.itp')
    print('✓ posre_backbone.itp generated (100 kJ/mol/nm² backbone only)')
else:
    print('⚠ posre_backbone.itp not generated — Step 18 will be skipped')

✓ posre.itp generated (1000 kJ/mol/nm² all heavy atoms)
✓ posre_backbone.itp generated (100 kJ/mol/nm² backbone only)


In [6]:
# Cell 4 — Write NVT MDP (Step 16)
nvt_mdp = """
; NVT Equilibration — 100 ps at 310 K
integrator  = md
nsteps      = 50000
dt          = 0.002

nstxout     = 500
nstvout     = 500
nstenergy   = 500
nstlog      = 500

cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0

coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16

constraints         = h-bonds
constraint_algorithm = lincs
lincs_iter          = 1
lincs_order         = 4

tcoupl      = V-rescale
tc-grps     = Protein    Non-Protein
tau_t       = 0.1        0.1
ref_t       = 310        310

pcoupl      = no

gen_vel     = yes
gen_temp    = 310
gen_seed    = -1

define      = -DPOSRES
"""
with open('/content/nvt.mdp', 'w') as f:
    f.write(nvt_mdp)
print('✓ nvt.mdp written')

✓ nvt.mdp written


In [7]:
# Cell 5 — grompp for NVT
result = subprocess.run([
    GMX, 'grompp',
    '-f', '/content/nvt.mdp',
    '-c', '/content/em.gro',
    '-r', '/content/em.gro',
    '-p', '/content/topol.top',
    '-o', '/content/nvt.tpr',
    '-maxwarn', '3'
], capture_output=True, text=True)

print(result.stderr[-2000:])
if result.returncode != 0:
    raise RuntimeError('grompp NVT failed — check errors above')
print('✓ grompp NVT complete — nvt.tpr generated')


      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            Teemu Murtola       
        Szilard Pall               Sander Pronk              Roland Schulz       
       Michael Shirts            Alexey Shvetsov             Alfons Sijbers      
       Peter Tieleman              Jon Vincent              Teemu Virolainen     
     Christian Wennberg            Maarten Wolf              Artem Zhmurov       
                           and the project leaders:
        Mark Abraham, Berk Hess, Erik Lindahl, and David van der Spoel

Copyright (c) 1991-2000, University of Groningen, The Netherlands.
Copyright (c) 2001-2019, The GROMACS development team at
Uppsala University, Stockholm University and
the Royal Institute of Technology, Sweden.
check out http://www.gromacs.org for more information.

GROMACS is free software; you can

In [8]:
# Cell 6 — Step 16: Run NVT (CPU, with checkpoint resume)
# If interrupted, just rerun this cell — it resumes from last checkpoint
print('Running NVT equilibration (100 ps)...')
print('Expected time: ~45-90 min on CPU')
print('If interrupted: rerun this cell to resume from checkpoint\n')

cpt = '/content/nvt.cpt'
nvt_gro = '/content/nvt.gro'

if nvt_gro and Path(nvt_gro).exists():
    print('✓ nvt.gro already exists — NVT already complete, skipping')
elif Path(cpt).exists():
    print('Checkpoint found — resuming NVT from last checkpoint...')
    result = subprocess.run([
        GMX, 'mdrun', '-v',
        '-deffnm', '/content/nvt',
        '-cpi', cpt,
        '-noappend'
    ], capture_output=True, text=True)
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
else:
    print('Starting fresh NVT run...')
    result = subprocess.run([
        GMX, 'mdrun', '-v',
        '-deffnm', '/content/nvt',
        '-ntmpi', '1',
        '-ntomp', '2'
    ], capture_output=True, text=True)
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    if result.returncode != 0:
        raise RuntimeError('mdrun NVT failed — check errors above')

print('\n✓ NVT complete')

Running NVT equilibration (100 ps)...
Expected time: ~45-90 min on CPU
If interrupted: rerun this cell to resume from checkpoint

Starting fresh NVT run...

26
step 45300, will finish Thu Apr  9 12:38:04 2026
step 45400, will finish Thu Apr  9 12:38:05 2026
step 45500, will finish Thu Apr  9 12:38:04 2026
step 45600, will finish Thu Apr  9 12:38:05 2026
step 45700, will finish Thu Apr  9 12:38:04 2026
step 45800, remaining wall clock time:   299 s          
step 45900, remaining wall clock time:   292 s          
step 46000, remaining wall clock time:   285 s          
step 46100, remaining wall clock time:   277 s          
step 46200, remaining wall clock time:   270 s          
step 46300, remaining wall clock time:   263 s          
step 46400, remaining wall clock time:   256 s          
step 46500, remaining wall clock time:   249 s          
step 46600, remaining wall clock time:   242 s          
step 46700, remaining wall clock time:   235 s          
step 46800, remaining wal

In [9]:
# Cell 7 — Check NVT temperature convergence
result = subprocess.run(
    f'echo "Temperature" | {GMX} energy -f /content/nvt.edr -o /content/nvt_temp.xvg',
    shell=True, capture_output=True, text=True
)
temp_matches = re.findall(r'Temperature\s+([\d.]+)\s+([\d.]+)', result.stderr)
if temp_matches:
    avg_temp, std_temp = temp_matches[-1]
    print(f'Average Temperature = {avg_temp} ± {std_temp} K')
    if 305 <= float(avg_temp) <= 315:
        print('✓ Temperature converged near 310 K')
    else:
        print('⚠ Temperature outside 305-315 K range')
else:
    print('✓ NVT complete — temperature data in nvt_temp.xvg')

✓ NVT complete — temperature data in nvt_temp.xvg


In [10]:
# Cell 8 — Write NPT MDP (Step 17)
npt_mdp = """
; NPT Equilibration — 100 ps at 310 K, 1 bar
integrator  = md
nsteps      = 50000
dt          = 0.002

nstxout     = 500
nstvout     = 500
nstenergy   = 500
nstlog      = 500

cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0

coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16

constraints         = h-bonds
constraint_algorithm = lincs
lincs_iter          = 1
lincs_order         = 4

tcoupl      = V-rescale
tc-grps     = Protein    Non-Protein
tau_t       = 0.1        0.1
ref_t       = 310        310

pcoupl          = Parrinello-Rahman
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5
refcoord_scaling = com

gen_vel     = no

define      = -DPOSRES
"""
with open('/content/npt.mdp', 'w') as f:
    f.write(npt_mdp)
print('✓ npt.mdp written')

✓ npt.mdp written


In [11]:
# Cell 9 — grompp for NPT
result = subprocess.run([
    GMX, 'grompp',
    '-f', '/content/npt.mdp',
    '-c', '/content/nvt.gro',
    '-r', '/content/nvt.gro',
    '-t', '/content/nvt.cpt',
    '-p', '/content/topol.top',
    '-o', '/content/npt.tpr',
    '-maxwarn', '3'
], capture_output=True, text=True)

print(result.stderr[-2000:])
if result.returncode != 0:
    raise RuntimeError('grompp NPT failed — check errors above')
print('✓ grompp NPT complete — npt.tpr generated')

 Shirts            Alexey Shvetsov             Alfons Sijbers      
       Peter Tieleman              Jon Vincent              Teemu Virolainen     
     Christian Wennberg            Maarten Wolf              Artem Zhmurov       
                           and the project leaders:
        Mark Abraham, Berk Hess, Erik Lindahl, and David van der Spoel

Copyright (c) 1991-2000, University of Groningen, The Netherlands.
Copyright (c) 2001-2019, The GROMACS development team at
Uppsala University, Stockholm University and
the Royal Institute of Technology, Sweden.
check out http://www.gromacs.org for more information.

GROMACS is free software; you can redistribute it and/or modify it
under the terms of the GNU Lesser General Public License
as published by the Free Software Foundation; either version 2.1
of the License, or (at your option) any later version.

GROMACS:      gmx grompp, version 2021.4-Ubuntu-2021.4-2
Executable:   /usr/bin/gmx
Data prefix:  /usr
Working dir:  /content
Comma

In [12]:
# Cell 10 — Step 17: Run NPT (CPU, with checkpoint resume)
print('Running NPT equilibration (100 ps)...')
print('Expected time: ~45-90 min on CPU')
print('If interrupted: rerun this cell to resume from checkpoint\n')

cpt = '/content/npt.cpt'
npt_gro = '/content/npt.gro'

if Path(npt_gro).exists():
    print('✓ npt.gro already exists — NPT already complete, skipping')
elif Path(cpt).exists():
    print('Checkpoint found — resuming NPT from last checkpoint...')
    result = subprocess.run([
        GMX, 'mdrun', '-v',
        '-deffnm', '/content/npt',
        '-cpi', cpt,
        '-noappend'
    ], capture_output=True, text=True)
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
else:
    print('Starting fresh NPT run...')
    result = subprocess.run([
        GMX, 'mdrun', '-v',
        '-deffnm', '/content/npt',
        '-ntmpi', '1',
        '-ntomp', '2'
    ], capture_output=True, text=True)
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    if result.returncode != 0:
        raise RuntimeError('mdrun NPT failed — check errors above')

print('\n✓ NPT complete')

Running NPT equilibration (100 ps)...
Expected time: ~45-90 min on CPU
If interrupted: rerun this cell to resume from checkpoint

Starting fresh NPT run...

00, will finish Thu Apr  9 13:45:54 2026
step 45600, will finish Thu Apr  9 13:45:53 2026
step 45700, will finish Thu Apr  9 13:45:54 2026
step 45800, will finish Thu Apr  9 13:45:53 2026
step 45900, remaining wall clock time:   298 s          
step 46000, remaining wall clock time:   290 s          
step 46100, remaining wall clock time:   283 s          
step 46200, remaining wall clock time:   276 s          
step 46300, remaining wall clock time:   269 s          
step 46400, remaining wall clock time:   261 s          
step 46500, remaining wall clock time:   254 s          
step 46600, remaining wall clock time:   247 s          
step 46700, remaining wall clock time:   239 s          
step 46800, remaining wall clock time:   232 s          
step 46900, remaining wall clock time:   225 s          
step 47000, remaining wall c

In [13]:
# Cell 11 — Check NPT pressure + density
for quantity in ['Pressure', 'Density']:
    result = subprocess.run(
        f'echo "{quantity}" | {GMX} energy -f /content/npt.edr -o /content/npt_{quantity.lower()}.xvg',
        shell=True, capture_output=True, text=True
    )
    matches = re.findall(rf'{quantity}\s+([\d.\-]+)\s+([\d.]+)', result.stderr)
    if matches:
        avg, std = matches[-1]
        print(f'{quantity}: avg = {avg}, std = {std}')
    else:
        print(f'✓ {quantity} saved to npt_{quantity.lower()}.xvg')

print('\nExpected: Pressure ~1 bar | Density ~1000 kg/m³')

✓ Pressure saved to npt_pressure.xvg
✓ Density saved to npt_density.xvg

Expected: Pressure ~1 bar | Density ~1000 kg/m³


In [14]:
# Cell 12 — Step 18: Restraint release (optional)
npt2_mdp = """
; NPT Restraint Release — 100 ps backbone only
integrator  = md
nsteps      = 50000
dt          = 0.002

nstxout     = 500
nstvout     = 500
nstenergy   = 500
nstlog      = 500

cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0

coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16

constraints         = h-bonds
constraint_algorithm = lincs

tcoupl      = V-rescale
tc-grps     = Protein    Non-Protein
tau_t       = 0.1        0.1
ref_t       = 310        310

pcoupl          = Parrinello-Rahman
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5
refcoord_scaling = com

gen_vel     = no
define      = -DPOSRES_BACKBONE
"""
with open('/content/npt2.mdp', 'w') as f:
    f.write(npt2_mdp)

result = subprocess.run([
    GMX, 'grompp',
    '-f', '/content/npt2.mdp',
    '-c', '/content/npt.gro',
    '-r', '/content/npt.gro',
    '-t', '/content/npt.cpt',
    '-p', '/content/topol.top',
    '-o', '/content/npt2.tpr',
    '-maxwarn', '3'
], capture_output=True, text=True)

if result.returncode == 0:
    print('✓ grompp npt2 complete')
    result2 = subprocess.run([
        GMX, 'mdrun', '-v',
        '-deffnm', '/content/npt2',
        '-ntmpi', '1', '-ntomp', '2'
    ], capture_output=True, text=True)
    print(result2.stdout[-2000:])
    print(result2.stderr[-1000:])
    print('\n✓ Restraint release complete' if result2.returncode == 0 else '\n✗ npt2 failed')
else:
    print('⚠ Step 18 skipped — proceed with npt.gro')
    print(result.stderr[-300:])

✓ grompp npt2 complete


step 49000, remaining wall clock time:    72 s          
step 49100, remaining wall clock time:    64 s          
step 49200, remaining wall clock time:    57 s          
step 49300, remaining wall clock time:    50 s          
step 49400, remaining wall clock time:    43 s          
step 49500, remaining wall clock time:    36 s          
step 49600, remaining wall clock time:    28 s          
step 49700, remaining wall clock time:    21 s          
step 49800, remaining wall clock time:    14 s          
step 49900, remaining wall clock time:     7 s          
Writing final coordinates.

step 50000, remaining wall clock time:     0 s          
               Core t (s)   Wall t (s)        (%)
       Time:     7221.559     3610.780      200.0
                         1h00:10
                 (ns/day)    (hour/ns)
Performance:        2.393       10.030

GROMACS reminds you: "Before we work on artificial intelligence why don't we do something about natural stup

In [15]:
# Cell 13 — Summary + Download
from google.colab import files

final_gro = '/content/npt2.gro' if Path('/content/npt2.gro').exists() else '/content/npt.gro'
final_cpt = '/content/npt2.cpt' if Path('/content/npt2.cpt').exists() else '/content/npt.cpt'
step18 = 'npt2.gro (done)' if 'npt2' in final_gro else 'skipped (optional)'

print('=' * 60)
print('  PHASE 3.3 STEP 3 - EQUILIBRATION COMPLETE (CPU)')
print('=' * 60)
print(f'  Step 16 - NVT (100 ps)      : nvt.gro @ 310 K')
print(f'  Step 17 - NPT (100 ps)      : npt.gro @ 310 K, 1 bar')
print(f'  Step 18 - Restraint release : {step18}')
print(f'  Final structure             : {Path(final_gro).name}')
print('=' * 60)
print('\nNext: Phase 3.3 Step 4 - Production MD')

for fpath in [final_gro, final_cpt, '/content/nvt.gro', '/content/npt.gro']:
    if Path(fpath).exists():
        files.download(fpath)
        print(f'✓ Downloaded {Path(fpath).name}')

  PHASE 3.3 STEP 3 - EQUILIBRATION COMPLETE (CPU)
  Step 16 - NVT (100 ps)      : nvt.gro @ 310 K
  Step 17 - NPT (100 ps)      : npt.gro @ 310 K, 1 bar
  Step 18 - Restraint release : npt2.gro (done)
  Final structure             : npt2.gro

Next: Phase 3.3 Step 4 - Production MD


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded npt2.gro


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✓ Downloaded nvt.gro


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded npt.gro
